# Notebook 4: Model Evaluation & Comparison

Compare:
- Frequentist logistic regression (sklearn)
- Bayesian logistic regression (PyMC posterior mean)
- Bayesian probit regression (PyMC posterior mean)

Metrics: ROC-AUC, accuracy, calibration, WAIC/LOO.

In [ ]:
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from scipy.special import expit
import sys; sys.path.append('..')
from src.preprocess import full_pipeline
from src.evaluate import classification_metrics, plot_roc_curves, plot_calibration, plot_posterior_coefs

X_train, X_test, y_train, y_test, scaler = full_pipeline('../data/framingham.csv')

# Load saved traces
idata_logistic = az.from_netcdf('../data/idata_logistic.nc')
idata_probit   = az.from_netcdf('../data/idata_probit.nc')

In [ ]:
# Frequentist baseline
freq_model = LogisticRegression(max_iter=1000, random_state=42)
freq_model.fit(X_train, y_train)
y_prob_freq = freq_model.predict_proba(X_test)[:, 1]

print('=== Frequentist Logistic Regression ===')
classification_metrics(y_test, y_prob_freq)

In [ ]:
# Bayesian logistic: use posterior mean of alpha & beta for prediction
alpha_post = idata_logistic.posterior['alpha'].values.mean()
beta_post  = idata_logistic.posterior['beta'].values.mean(axis=(0, 1))
y_prob_bayes_logistic = expit(X_test @ beta_post + alpha_post)

print('=== Bayesian Logistic Regression ===')
classification_metrics(y_test, y_prob_bayes_logistic)

In [ ]:
# Bayesian probit: Phi(eta)
from scipy.stats import norm
alpha_p = idata_probit.posterior['alpha'].values.mean()
beta_p  = idata_probit.posterior['beta'].values.mean(axis=(0, 1))
y_prob_bayes_probit = norm.cdf(X_test @ beta_p + alpha_p)

print('=== Bayesian Probit Regression ===')
classification_metrics(y_test, y_prob_bayes_probit)

In [ ]:
# ROC & Calibration comparison
models = {
    'Frequentist LR': y_prob_freq,
    'Bayesian Logistic': y_prob_bayes_logistic,
    'Bayesian Probit': y_prob_bayes_probit,
}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_roc_curves(models, y_test, ax=axes[0])
plot_calibration(models, y_test, ax=axes[1])
plt.tight_layout()

In [ ]:
# Posterior coefficient forest plot
plot_posterior_coefs(idata_logistic, feature_names=None, model_name='Bayesian Logistic')

In [ ]:
# WAIC model comparison
# (requires log-likelihood to be stored — see PyMC docs for idata_kwargs)
# az.compare({'logistic': idata_logistic, 'probit': idata_probit}, ic='waic')